# Cleaning RIPA Datasets
Based off of https://github.com/joshuagrossman/ripa/tree/main/src/ripa cleaning scripts

In [ ]:
import requests # To download files from the internet
import zipfile # To open zip files
import io # To treat downloaded data like a file in memory
import pandas as pd


def load_ripa_from_doj_zip(county):
    """
    Downloads DOJ RIPA Stop Data (2019–2023),
    loads only the Excel file that contains the county name,
    and returns a combined DataFrame.
    
    Example:
        df = load_ripa_from_doj_zip("Orange")
    """

    # DOJ public ZIP URLs for 2019–2023
    url_dict = {
        2019: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2019.zip",
        2020: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2020.zip",
        2021: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2021.zip",
        2022: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2022.zip",
        2023: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2023.zip",
    }

    all_years = []

    for year, zip_url in url_dict.items():

        response = requests.get(zip_url)
        response.raise_for_status() # Checks whether website request succeeded

        z = zipfile.ZipFile(io.BytesIO(response.content)) # Opens the zip in memory

        # Find the Excel file containing the county name
        county_file = None # Creates placeholder value
        for file in z.namelist(): # For every file in the ZIP
            if county.lower() in file.lower() and file.lower().endswith(".xlsx"): # Check if file is the right county
                county_file = file
                break

        # If there's no file for that county
        if county_file is None:
            raise ValueError(f"No file found for {county} in {year}")

        # Open the file
        with z.open(county_file) as f:
            df_year = pd.read_excel(f, engine="openpyxl")

        # Make all columns lower-case strings
        df_year.columns = df_year.columns.str.lower()
        df_year["year"] = year # Add year column

        all_years.append(df_year)

    final_df = pd.concat(all_years, ignore_index=True)

    return final_df


In [10]:
load_ripa_from_doj_zip("Orange")

<positron-console-cell-10>:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


,doj_record_id,person_number,agency_ori,agency_name,time_of_stop,date_of_stop,stop_duration,closest_city,school_code,school_name,stop_student,k12_school_grounds,rae_full,rae_asian,rae_black_african_american,rae_hispanic_latino,rae_middle_eastern_south_asian,rae_native_american,rae_pacific_islander,rae_white,rae_multiracial,g_full,g_male,g_female,g_transgender_man,g_transgender_woman,g_gender_nonconforming,g_multigender,lgbt,age,age_group,limited_english_fluency,pd_full,pd_deafness_hearing,pd_speech_impair,pd_blind,pd_mental_health,pd_devel_disab,pd_hyperactivity_disability,pd_other,...,ced_money,ced_drug_paraphernalia,ced_stolen_prop,ced_elect_device,ced_other_contraband,bps_safekeeping,bps_contraband,bps_evidence,bps_impound_vehicle,bps_abandon_prop,bps_violate_school,tps_firearm,tps_ammunition,tps_weapon,tps_drugs,tps_alcohol,tps_money,tps_drug_paraphernalia,tps_stolen_prop,tps_cellphone,tps_vehicle,tps_contraband,ros_no_action,ros_warning,ros_citation,ros_in_field_cite_release,ros_custodial_warrant,ros_custodial_without_warrant,ros_field_interview_card,ros_noncriminal_transport,ros_contact_legal_guardian,ros_psych_hold,ros_us_homeland,ros_referral_school_admin,ros_referral_school_counselor,ros_warning_cds,ros_citation_cds,ros_in_field_cite_release_cds,ros_custodial_wout_warrant_cds,year
0,W300020069A6O99XAGK5,1,CA0300000,ORANGE CO SO,1117,2019-01-26,10,MISSION VIEJO,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,55,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54106,NaN,NaN,2020
1,W300020069D4XYP7190K,1,CA0300000,ORANGE CO SO,2030,2019-01-26,30,RANCHO SANTA MARGARITA,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,25,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54109,NaN,NaN,2020
2,W30002006972FIDUZ3KR,1,CA0300000,ORANGE CO SO,1545,2019-01-26,30,MIDWAY CITY,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,30,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,0,0,0,0,0,0,0,0,0,32022,NaN,NaN,NaN,2020
3,W300020069POLZN9EMJE,1,CA0300000,ORANGE CO SO,724,2019-01-26,10,MISSION VIEJO,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,40,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54106,NaN,NaN,2020
4,W300020069Q9PU5JLJHF,1,CA0300000,ORANGE CO SO,1942,2019-01-26,20,BUENA PARK,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,30,5,0,4,0,0,0,1,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50391,W300020085YXP1YUYYB3,1,CA0300000,ORANGE CO SO,750,2019-05-20,5,LAKE FOREST,NaN,NaN,0,0,8,1,0,1,0,1,0,1,1,2,0,1,0,0,0,0,0,45,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54167,NaN,NaN,2020
50392,W3000200852HWB7Q8NGE,1,CA0300000,ORANGE CO SO,1734,2019-06-12,5,LAGUNA NIGUEL,NaN,NaN,0,0,8,0,0,1,1,0,1,1,1,1,1,0,0,0,0,0,0,33,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54098,NaN,NaN,2020
50393,W3000200801IXQ455J0E,1,CA0300000,ORANGE CO SO,758,2019-01-08,6,LAKE FOREST,NaN,NaN,0,0,8,1,0,1,1,1,1,1,1,2,0,1,0,0,0,0,0,25,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54104,NaN,NaN,2020
50394,W300020086JQY1LWL0P9,1,CA0300000,ORANGE CO SO,1200,2019-02-25,5,LAKE FOREST,NaN,NaN,0,0,8,1,0,1,1,1,1,1